In [1]:
import json
from pathlib import Path

import pandas as pd

**Роль baseline.** Контрольный эксперимент: из него сознательно исключен весь класс
агрегационных признаков (membership / frequency / «seen before» / агрегаты по
сущностям) — именно они являются исследуемым эффектом этапов `bloom` и `exact`.
Поэтому baseline заведомо слабее типового решения этого соревнования; его задача —
дать честную точку отсчета, а не максимальную метрику.

# 0. Загрузка артефактов

In [2]:
REPORTS = Path('../reports')

av = pd.read_csv(REPORTS / 'av_d_features.csv')

run_dir = sorted((REPORTS / 'runs').glob('baseline_*'))[-1]
metrics = json.loads((run_dir / 'metrics.json').read_text())

print('прогон:', run_dir.name)

прогон: baseline_20260727_160710


# 1. Отбор D-признаков: результаты adversarial validation

Правило отбора (02_feature_eda, §3): для каждой D-колонки однопризнаковый
классификатор train-vs-test прогоняется на обеих версиях — сырой и нормализованной;
в модель идет версия с AUC $\le$ 0.55, при дрейфе обеих версий колонка исключается.

In [3]:
av.sort_values('column', key=lambda s: s.str.lstrip('D').astype(int))

,column,version,auc,verdict
0,D1,raw,0.539237,pass
1,D1,norm,0.914119,fail
2,D2,raw,0.533462,pass
3,D2,norm,0.723316,fail
4,D3,raw,0.530248,pass
5,D3,norm,0.882256,fail
6,D4,raw,0.624681,fail
7,D4,norm,0.819270,fail
8,D5,raw,0.550223,fail
9,D5,norm,0.837411,fail


* Сводно по колонкам:

| колонка | auc_raw | auc_norm | решение |
|---|---|---|---|
| D1 | 0.539 | 0.914 | **raw** |
| D2 | 0.533 | 0.723 | **raw** |
| D3 | 0.530 | 0.882 | **raw** |
| D4 | 0.625 | 0.819 | excluded |
| D5 | 0.550 | 0.837 | excluded |
| D6 | 0.565 | 0.636 | excluded |
| D7 | 0.527 | 0.573 | **raw** |
| D8 | 0.516 | 0.583 | **raw** |
| D9 | 0.510 | 0.627 | excluded — отдельное решение: дубликат `TransactionHour` (§2) |
| D10 | 0.630 | 0.870 | excluded |
| D11 | 0.594 | 0.729 | excluded |
| D12 | 0.516 | 0.594 | **raw** |
| D13 | 0.571 | 0.651 | excluded |
| D14 | 0.564 | 0.631 | excluded |
| D15 | 0.657 | 0.850 | excluded |

**Итог:** в модель идут сырые D1, D2, D3, D7, D8, D12. Ни одна `Norm`-версия порог
не прошла. Суммарная различимость train/test по D-блоку: **0.776 до отбора
$\to$ 0.588 после**.

Фактическая резолюция зафиксирована в `configs/base.yaml`, блок `d_features`
(`status: applied_from_av`); пайплайн читает ее оттуда и AV не пересчитывает —
решения детерминированы и версионируются вместе с конфигом.

## 1.1. Расхождение с ожиданием по EDA

Ожидание из 02_feature_eda §3 (по графику «лесенка / полка»): `Norm` для D1, D4, D10,
D11, D15. Результат оказался **обратным** — для всех пяти колонок сырая версия ближе
к порогу, чем нормализованная, причем с большим отрывом.

Проверка на данных (средние значения):

| | train | test |
|---|---|---|
| D1 | 94.35 | 108.21 |
| D1Norm | 9.71 | −202.47 |

**Вывод:** сырой D1 на уровне *распределения* оказался почти стационарен (сдвиг ~14
при std $\approx$ 160), поэтому $D1Norm = D1 - TransactionDay$ вырождается в
календарный тренд: значения test целиком лежат вне train-диапазона, и все тестовые
объекты попадают в один крайний лист дерева. Аргумент против нормализации гэп-признаков
(§3) применим и к D1.

Две причины, по которым визуальная EDA дала неверный прогноз:
1. вывод строился по траекториям отдельных карт (график «лесенка / полка», 02_feature_eda
   §3): у фиксированной сущности сырой D1 растет на +1 день/день. Но рост внутри сущности
   не означает сдвига распределения по популяции: при постоянном притоке новых сущностей
   распределение «возраста на момент транзакции» может оставаться стационарным — что
   и произошло (сдвиг средних train/test ~14 при std $\approx$ 160);
2. постоянство `Norm` внутри сущности не помогает *числовому* признаку: CatBoost строит
   сплиты по значению, а не по сущности, и стабильность внутри сущности не спасает,
   если весь диапазон вышел за пределы обучающей поддержки.

Это ровно тот случай, ради которого отбор был автоматизирован: формальная процедура
опровергла гипотезу, основанную на визуальном осмотре траекторий трех карт.

**Что сохраняется:** наблюдение о «полках» остается в силе и используется дальше —
`D1Norm` может быть полезен в композитном ключе `card1 + addr1 + D1Norm` как **компонент
категориального идентификатора** для фильтров Блума, где выход за числовой диапазон
нерелевантен.

# 2. Структура валидации

Схема зафиксирована в 02_feature_eda §6 и заморожена для всех экспериментов:
сплиты обязаны быть идентичными, иначе сравнение экспериментов невалидно.

* **Holdout:** граница по дню 141 (последние ~20% train по времени), 472 006 / 118 534.
* **Скользящий CV:** три полных фолда после корректировки (неполный последний блок
  отброшен, старт фолдов сдвинут на блок раньше).

In [4]:
folds = pd.DataFrame(metrics['cv']['folds'])
folds[['fold', 'train_rows', 'valid_rows', 'best_iteration', 'auc', 'seconds']]

,fold,train_rows,valid_rows,best_iteration,auc,seconds
0,0,312574,98027,208,0.899925,175.592166
1,1,410601,85303,1312,0.927843,836.344253
2,2,495904,86525,761,0.923776,628.651567


# 3. Метрики baseline

In [5]:
cv = metrics['cv']
ho = metrics['holdout']

print(f"CV mean AUC : {cv['auc_mean']:.5f} ± {cv['auc_std']:.5f}")
for f in cv['folds']:
    print(f"  fold {f['fold']}: AUC={f['auc']:.5f}")
print(f"\nHoldout AUC : {ho['auc']:.5f}  (best_iter={ho['best_iteration']})")

print("\nPublic LB AUC : 0.92663")
print("\nPrivate LB AUC : 0.89437")

CV mean AUC : 0.91718 ± 0.01508
  fold 0: AUC=0.89992
  fold 1: AUC=0.92784
  fold 2: AUC=0.92378

Holdout AUC : 0.91551  (best_iter=404)

Public LB AUC : 0.92663

Private LB AUC : 0.89437


## 3.1. Интерпретация

**Согласованность оценок.** CV-mean (0.9172), holdout (0.9155) и public LB (0.9266)
лежат близко; private LB (0.8944) заметно ниже. Убывание CV $\to$ holdout $\to$
private LB отражает временной дрейф: чем дальше валидационный период от обучающего,
тем ниже качество. Разрыв public / private (0.032) — прямая количественная иллюстрация
того же явления, с которым работали отбор D-признаков и запрет сырых временных
признаков.

**Фолд 0 системно сложнее остальных** (0.8999 против 0.9278 и 0.9238) — это свойство
схемы, а не шум: он обучается на наименьшем объеме (312 тыс. против 411 и 496 тыс.)
и на самом раннем участке данных, , где популяция сущностей еще не устоялась после
начала окна наблюдения.

**Следствие для методики.** Величина $\pm$ 0.015 характеризует различие между
*временными периодами*, а не точность измерения эффекта. Поэтому эксперименты
сравниваются попарно по фолдам ($\Delta_{fold}$, см. 02_feature_eda §7): систематическая
сложность периода одинакова у сравниваемых экспериментов и при вычитании сокращается.

**Уровень качества.** Достигнутый private LB ниже типичных решений соревнования
(победитель — около 0.946), что ожидаемо: из baseline исключен весь класс агрегационных
признаков по сущностям, дающий на этом датасете основной прирост.

# 4. Ресурсы

Точка отсчета для раздела об эффективности.

In [6]:
for stage in ('cv', 'holdout', 'submit'):
    m = metrics[stage]
    print(f"{stage:<8} {m['seconds']:>8.0f} с   "
          f"{m['peak_memory_bytes'] / 1024**2:>8.0f} МиБ")

cv           1641 с       9751 МиБ
holdout       383 с       8262 МиБ
submit        367 с       7761 МиБ


| этап | время | пиковая память |
|---|---|---|
| CV (3 фолда) | 1 641 с (~27 мин) | 9 751 МиБ |
| Holdout | 383 с | 8 261 МиБ |
| Переобучение + сабмит | 367 с (465 итераций) | 7 761 МиБ |

**Важная оговорка.** Приведенные цифры характеризуют ресурсоемкость *обучения модели*
(pandas + CatBoost на матрице 590 540 $\times$ ~460) и не являются метрикой для главы
об эффективности фильтров Блума. Там измеряется другой объект — **размер самой
структуры данных в байтах** (фильтр Блума против точного множества на том же наборе
ключей), фактический FPR и скорость операций вставки/запроса. Соответствующие замеры
закладываются на этапе `bloom` отдельно.

# 5. Итоги этапа

1. Отбор D-признаков автоматизирован и выполнен: в модель идут сырые D1, D2, D3, D7,
   D8, D12; суммарная различимость train/test по D-блоку снижена с 0.776 до 0.588.
   Гипотеза о полезности нормализации, выдвинутая по визуальной EDA, формально
   опровергнута.
2. Схема валидации зафиксирована и заморожена: holdout по дню 141 + три полных
   скользящих фолда сопоставимого размера.
3. Baseline обучен: CV 0.91718 $\pm$ 0.01508, holdout 0.91551, private LB 0.89437.
4. Зафиксирована точка отсчета по ресурсам обучения.

**Следующий этап** — построение признаков на фильтрах Блума (`bloom`) и их точных
аналогах (`exact`) поверх этого же пайплайна: базовые признаки, гиперпараметры и сплиты
остаются неизменными, эксперименты различаются только блоком `extra_features` в своем
конфиге.